In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import random

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, MultiProductContextEmbeddings
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [ ]:
BASE_SEED = 42

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Ajustes

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = RESULTS_DIR / "checkpoints" / "final_eval"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_TRIAL_PATH = RESULTS_DIR / "best_trial_params.json"

# ── Datos / evaluación ─────────────────────────────────────────────
N_UPCS = 5
SMOOTH_WINDOW = 8
BETA_EDA = -2

TRAIN_FRAC = 0.8          # split final para bootstrap
N_FOLDS = 5               # evaluación temporal final
EVAL_SEEDS = [11, 29, 42, 77, 123]
N_BOOTSTRAP = 100
BLOCK_SIZE = 4
MIN_TRAIN_FRAC = 0.5

# ── Entrenamiento ──────────────────────────────────────────────────
N_EPOCHS_P0 = 350
N_EPOCHS_P1 = 350
N_EPOCHS_P2 = 400
PATIENCE    = 30
ES_PATIENCE = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Cargar mejor trial ─────────────────────────────────────────────
with open(BEST_TRIAL_PATH, "r", encoding="utf-8") as f:
    best_trial = json.load(f)

HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64_32":  (128, 64, 32),
    "64_32_16":   (64, 32, 16),
    "128_64":     (128, 64),
}

params = best_trial["params"]

N_KNOTS = int(params["N_KNOTS"])
HIDDEN = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
ACT = "gelu"
DROPOUT = float(params["DROPOUT"])
D_STORE = 16

BATCH_SIZE = int(params["BATCH_SIZE"])
LR_P0 = float(params["LR_P0"])
LR_P1 = float(params["LR_P1"])
LR_P2 = float(params["LR_P2"])
LAMBDA_SMOOTH_P2 = float(params["LAMBDA_SMOOTH_P2"])
LAMBDA_POS_P2 = float(params["LAMBDA_POS_P2"])

print("Best trial cargado:")
print(json.dumps(best_trial, indent=2, ensure_ascii=False))

Device: cuda


# Cargar y Preparar Datos

In [ ]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")

Dataset shape: (463722, 30)
Columnas: ['store_code', 'upc_code', 'category_code', 'week_id', 'log_liters_sold', 'log_price_per_liter', 'on_promo', 'week_rank', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'sin_13', 'cos_13', 'weeks_since_first_seen_upc', 'weeks_since_first_seen_store_upc', 'liters_per_upc', 'lag_1_log_liters_sold', 'lag_2_log_liters_sold', 'lag_4_log_liters_sold', 'rolling_mean_4_log_liters_sold', 'rolling_mean_8_log_liters_sold', 'rolling_mean_13_log_liters_sold', 'miss_lag_1', 'miss_lag_2', 'miss_lag_4', 'miss_roll_4', 'miss_roll_8', 'miss_roll_13', 'promo_intensity_store_week']


# Factorizar colmnas Categóricas

In [ ]:
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id",    sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
print(f"Tiendas: {n_stores}  |  Semanas: {n_weeks}")

Tiendas: 70  |  Semanas: 302


In [ ]:
mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)

full_wide_raw = mp_builder.transform(df).copy()
n_upcs = mp_builder.n

store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs seleccionados: {n_upcs}")
print(f"Top {N_UPCS}: {mp_builder.selected_upcs[:N_UPCS]}")

In [ ]:
splitter = TemporalSplitter(week_col="week_id")
bootstrap_sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=BLOCK_SIZE,
    rng=np.random.default_rng(BASE_SEED),
)

fold_splits = splitter.expanding_splits(
    df=full_wide_raw,
    n_folds=N_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

print(f"N folds temporales: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i} | train={len(train_fold):,} ({train_fold['week_id'].nunique()} semanas) "
        f"| val={len(val_fold):,} ({val_fold['week_id'].nunique()} semanas)"
    )

In [ ]:
def prepare_fold_frames(train_wide: pd.DataFrame, val_wide: pd.DataFrame):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=SMOOTH_WINDOW, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s


def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s):
    loader_factory = DataLoaderFactory(num_workers=0, pin_memory=True)

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader_p0 = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=BATCH_SIZE, shuffle=False
    )

    train_loader = loader_factory.create_train_loader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader = loader_factory.create_eval_loader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False
    )

    return train_loader_p0, val_loader_p0, train_loader, val_loader

In [ ]:
def build_model_components(train_wide: pd.DataFrame):
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=N_KNOTS, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    context_builder = MultiProductContextEmbeddings(
        n=n_upcs,
        n_stores=n_stores,
        d_store=D_STORE,
    )

    def make_model(enforce_negative_beta, use_cross, device):
        head = IntegrableDemandHead(
            context_dim=context_builder.out_dim,
            K_splines=N_KNOTS,
            n=n_upcs,
            hidden=HIDDEN,
            act=ACT,
            dropout=DROPOUT,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        model = ICDN(
            context_builder=context_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)
        return model

    return make_model

In [ ]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, phase_name=""):

    best_val_loss = float("inf")
    no_improve    = 0
    history       = {"train_loss": [], "val_loss": [], "val_mae": [], "lr": []}
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    print(f"\n{'='*70}")
    print(f"  {phase_name}  |  {n_epochs} épocas  |  ckpt → {ckpt_path.name}")
    print(f"{'='*70}")

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, eps_hat, aux = model(batch, return_parts=True)
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                        aux["Bx"], aux["IBx"],
                                        model.head.param_head._pairs)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, eps_hat, aux = model(batch, return_parts=True)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                    aux["Bx"], aux["IBx"],
                                    model.head.param_head._pairs)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        train_loss = total_loss / max(total_denom, 1.0)

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        total_loss, total_denom = 0.0, 0.0
        total_abs,  total_mask  = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
                obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

                y_hat, eps_hat, aux = model(batch, return_parts=True)
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                aux["Bx"], aux["IBx"],
                                model.head.param_head._pairs)

                denom = obs_mask.sum().item()
                total_loss  += logs["loss"].item() * denom
                total_denom += denom

                total_abs  += ((y_hat - y_true).abs() * obs_mask).sum().item()
                total_mask += obs_mask.sum().item()

        val_loss = total_loss / max(total_denom, 1.0)
        val_mae  = total_abs  / max(total_mask, 1.0)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
            saved = "✓"
        else:
            no_improve += 1
            saved = ""

        if (epoch + 1) % 10 == 0 or no_improve == 0:
            print(f"Epoch {epoch+1:4d}/{n_epochs}"
                  f"  train={train_loss:.4f}"
                  f"  val={val_loss:.4f}"
                  f"  mae={val_mae:.4f}"
                  f"  {saved}")

        if no_improve >= es_patience:
            print(f"  Early stopping en época {epoch+1}")
            break

    print(f"\nMejor val_loss: {best_val_loss:.4f}  →  {ckpt_path.name}")
    return history

Helpers definidos


In [ ]:
def freeze_spline(model):
    with torch.no_grad():
        model.head.param_head.head_w.weight.zero_()
        model.head.param_head.head_w.bias.zero_()
    model.head.param_head.head_w.weight.requires_grad_(False)
    model.head.param_head.head_w.bias.requires_grad_(False)


def unfreeze_spline(model):
    model.head.param_head.head_w.weight.requires_grad_(True)
    model.head.param_head.head_w.bias.requires_grad_(True)


def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0
    )
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)
    print(f"β inicializado: target={beta_target:.3f}  beta_raw_init={beta_raw_init:.4f}")

print("Helpers definidos")

In [ ]:
def compute_global_metrics(model, val_loader):
    model.eval()
    all_y_hat, all_y_true, all_mask = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            y_hat, _, _ = model(batch, return_parts=True)
            y_true = torch.stack([batch[f"log_liters_{i}"] for i in range(n_upcs)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"] for i in range(n_upcs)], dim=1)

            all_y_hat.append(y_hat.cpu())
            all_y_true.append(y_true.cpu())
            all_mask.append(obs_mask.cpu())

    y_hat = torch.cat(all_y_hat)
    y_true = torch.cat(all_y_true)
    mask = torch.cat(all_mask)

    mae = float(((y_hat - y_true).abs() * mask).sum() / mask.sum())
    rmse = float(torch.sqrt((((y_hat - y_true) ** 2) * mask).sum() / mask.sum()))

    preds_np = y_hat.numpy()
    targets_np = y_true.numpy()
    mask_np = mask.bool().numpy()

    ss_res = ((targets_np[mask_np] - preds_np[mask_np]) ** 2).sum()
    ss_tot = ((targets_np[mask_np] - targets_np[mask_np].mean()) ** 2).sum()
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }


def extract_elasticity_rows(model, val_loader, run_type, run_id, fold=None, seed=None, bootstrap_run=None):
    model.eval()
    rows = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            store_idx = batch["store_code"].cpu().numpy()
            store_code = np.array(store_cats)[store_idx]

            obs_mask = torch.stack(
                [batch[f"obs_mask_{i}"] for i in range(n_upcs)], dim=1
            )

            y_hat, eps_hat, aux = model.run(batch, return_parts=True, compute_E=True)
            E = aux["E"].cpu().numpy()
            mask_np = obs_mask.cpu().numpy().astype(bool)
            y_true_np = torch.stack(
                [batch[f"log_liters_{i}"] for i in range(n_upcs)], dim=1
            ).cpu().numpy()
            y_hat_np = y_hat.cpu().numpy()

            B = E.shape[0]
            upc_names = np.array(mp_builder.selected_upcs)

            for b in range(B):
                sc = store_code[b]
                for i in range(n_upcs):
                    if not mask_np[b, i]:
                        continue
                    for j in range(n_upcs):
                        if not mask_np[b, j]:
                            continue
                        rows.append({
                            "run_type": run_type,
                            "run_id": run_id,
                            "fold": fold,
                            "seed": seed,
                            "bootstrap_run": bootstrap_run,
                            "store_code": sc,
                            "upc_i": upc_names[i],
                            "upc_j": upc_names[j],
                            "tipo": "own" if i == j else "cross",
                            "E": E[b, i, j],
                            "y_true_i": y_true_np[b, i],
                            "y_hat_i": y_hat_np[b, i],
                        })

    return pd.DataFrame(rows)

In [ ]:
def train_one_run(train_fold, val_fold, seed, run_type, run_id, fold=None, bootstrap_run=None):
    set_all_seeds(seed)

    train_wide, val_wide, train_wide_s, val_wide_s = prepare_fold_frames(train_fold, val_fold)
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s
    )

    make_model = build_model_components(train_wide)

    ckpt_p0 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase1.pt"
    ckpt_p2 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase2.pt"

    # Fase 0
    model_p0 = make_model(enforce_negative_beta=True, use_cross=False, device=device)
    freeze_spline(model_p0)
    init_beta_prior(model_p0, BETA_EDA)

    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in model_p0.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    run_training(
        model=model_p0,
        train_loader=train_loader_p0,
        val_loader=val_loader_p0,
        loss_fn=loss_p0,
        optimizer=opt_p0,
        scheduler=sch_p0,
        n_epochs=N_EPOCHS_P0,
        es_patience=ES_PATIENCE,
        ckpt_path=ckpt_p0,
        device=device,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P0",
    )

    # Fase 1
    model_p1 = make_model(enforce_negative_beta=True, use_cross=False, device=device)
    model_p1.load_state_dict(torch.load(ckpt_p0, map_location=device))

    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in model_p1.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    run_training(
        model=model_p1,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_p1,
        optimizer=opt_p1,
        scheduler=sch_p1,
        n_epochs=N_EPOCHS_P1,
        es_patience=ES_PATIENCE,
        ckpt_path=ckpt_p1,
        device=device,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P1",
    )

    # Fase 2
    model_p2 = make_model(enforce_negative_beta=True, use_cross=True, device=device)
    state = torch.load(ckpt_p1, map_location=device)
    state.pop("head.param_head._pairs", None)
    model_p2.load_state_dict(state, strict=False)
    unfreeze_spline(model_p2)

    with torch.no_grad():
        model_p2.head.param_head.head_cross.weight.zero_()
        model_p2.head.param_head.head_cross.bias.zero_()

    loss_p2 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=LAMBDA_SMOOTH_P2,
        lambda_pos=LAMBDA_POS_P2,
        reduction="none",
    )
    decay, no_decay = [], []
    for name, p in model_p2.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p2 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P2,
    )
    sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p2, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    run_training(
        model=model_p2,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_p2,
        optimizer=opt_p2,
        scheduler=sch_p2,
        n_epochs=N_EPOCHS_P2,
        es_patience=ES_PATIENCE,
        ckpt_path=ckpt_p2,
        device=device,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P2",
    )

    model_p2.load_state_dict(torch.load(ckpt_p2, map_location=device))

    metrics = compute_global_metrics(model_p2, val_loader)
    df_e = extract_elasticity_rows(
        model=model_p2,
        val_loader=val_loader,
        run_type=run_type,
        run_id=run_id,
        fold=fold,
        seed=seed,
        bootstrap_run=bootstrap_run,
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    ckpt_p2.unlink(missing_ok=True)

    return metrics, df_e

In [ ]:
fold_metrics_rows = []
fold_elasticity_rows = []

for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
    for seed in EVAL_SEEDS:
        print(f"\n=== Fold {fold_id} | Seed {seed} ===")

        metrics, df_e = train_one_run(
            train_fold=train_fold,
            val_fold=val_fold,
            seed=seed,
            run_type="kfold",
            run_id=f"fold{fold_id}_seed{seed}",
            fold=fold_id,
            bootstrap_run=None,
        )

        fold_metrics_rows.append({
            "fold": fold_id,
            "seed": seed,
            "n_train": len(train_fold),
            "n_val": len(val_fold),
            **metrics,
        })

        fold_elasticity_rows.append(df_e)

nn_kfold_metrics_raw = pd.DataFrame(fold_metrics_rows)
nn_kfold_elasticities_raw = pd.concat(fold_elasticity_rows, ignore_index=True)

print("K-fold + seeds completado")
display(nn_kfold_metrics_raw.head())

In [ ]:
nn_kfold_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_kfold_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_kfold_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_kfold_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_kfold_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_kfold_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_kfold_metrics_raw["r2_val"].std(ddof=1),
    "n_runs": len(nn_kfold_metrics_raw),
}])

display(nn_kfold_metrics_summary)

In [ ]:
train_final, val_final = splitter.single_split(full_wide_raw, train_frac=TRAIN_FRAC)
train_weeks_final = sorted(train_final["week_id"].unique())

print(f"Train final: {len(train_final):,} filas | {len(train_weeks_final)} semanas")
print(f"Val final:   {len(val_final):,} filas | {val_final['week_id'].nunique()} semanas")

In [ ]:
BOOTSTRAP_TRAINING_SEED = EVAL_SEEDS[0]

bootstrap_metrics_rows = []
bootstrap_elasticity_rows = []

for b in range(N_BOOTSTRAP):
    print(f"\n=== Bootstrap {b+1}/{N_BOOTSTRAP} ===")

    train_bs = bootstrap_sampler.sample(train_final, train_weeks_final)

    metrics, df_e = train_one_run(
        train_fold=train_bs,
        val_fold=val_final,
        seed=BOOTSTRAP_TRAINING_SEED,
        run_type="bootstrap",
        run_id=f"bootstrap{b}",
        fold=None,
        bootstrap_run=b,
    )

    bootstrap_metrics_rows.append({
        "bootstrap_run": b,
        "seed": BOOTSTRAP_TRAINING_SEED,
        "n_train": len(train_bs),
        "n_val": len(val_final),
        **metrics,
    })

    bootstrap_elasticity_rows.append(df_e)

nn_bootstrap_metrics_raw = pd.DataFrame(bootstrap_metrics_rows)
nn_bootstrap_elasticities_raw = pd.concat(bootstrap_elasticity_rows, ignore_index=True)

print("Bootstrap completado")
display(nn_bootstrap_metrics_raw.head())

In [ ]:
def q025(x): return np.percentile(x, 2.5)
def q975(x): return np.percentile(x, 97.5)

nn_bootstrap_own_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["tipo"] == "own"]
    .groupby(["store_code", "upc_i"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
    .rename(columns={"upc_i": "upc_code"})
)

nn_bootstrap_cross_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["tipo"] == "cross"]
    .groupby(["store_code", "upc_i", "upc_j"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
)

display(nn_bootstrap_own_summary.head())
display(nn_bootstrap_cross_summary.head())

In [ ]:
nn_bootstrap_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_bootstrap_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_bootstrap_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_bootstrap_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_bootstrap_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_bootstrap_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_bootstrap_metrics_raw["r2_val"].std(ddof=1),
    "n_bootstrap_runs": len(nn_bootstrap_metrics_raw),
}])

display(nn_bootstrap_metrics_summary)

In [ ]:
nn_kfold_metrics_raw.to_csv(DATA_DIR / "nn_kfold_metrics_raw.csv", index=False)
nn_kfold_metrics_summary.to_csv(DATA_DIR / "nn_kfold_metrics_summary.csv", index=False)
nn_kfold_elasticities_raw.to_csv(DATA_DIR / "nn_kfold_elasticities_raw.csv", index=False)

nn_bootstrap_metrics_raw.to_csv(DATA_DIR / "nn_bootstrap_metrics_raw.csv", index=False)
nn_bootstrap_metrics_summary.to_csv(DATA_DIR / "nn_bootstrap_metrics_summary.csv", index=False)
nn_bootstrap_elasticities_raw.to_csv(DATA_DIR / "nn_bootstrap_elasticities_raw.csv", index=False)
nn_bootstrap_own_summary.to_csv(DATA_DIR / "nn_bootstrap_own_summary.csv", index=False)
nn_bootstrap_cross_summary.to_csv(DATA_DIR / "nn_bootstrap_cross_summary.csv", index=False)

print("Guardado:")
print("- nn_kfold_metrics_raw.csv")
print("- nn_kfold_metrics_summary.csv")
print("- nn_kfold_elasticities_raw.csv")
print("- nn_bootstrap_metrics_raw.csv")
print("- nn_bootstrap_metrics_summary.csv")
print("- nn_bootstrap_elasticities_raw.csv")
print("- nn_bootstrap_own_summary.csv")
print("- nn_bootstrap_cross_summary.csv")